In [1]:
import subprocess, os

# ── 1. 修复 torchvision ────────────────────────────────────────────────
# ❌ 直接删掉或注释掉这一段！CloudGPU 预装的 torch 和 torchvision 是完美匹配的。
# ───────────────────────────────────────────────────────────────────────

print("✅ 跳过重装 torchvision，使用 CloudGPU 默认环境")

# ── 2. 克隆 kohya ─────────────────────────────────────────────────────
if not os.path.exists("/workspace/working/sd-scripts"):
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/kohya-ss/sd-scripts.git",
        "/workspace/working/sd-scripts"], check=True)
print("✅ sd-scripts 就绪")

# ── 3. 安装 kohya 依赖（跳过 torch 系列 / transformers / -e .）─────────
with open("/workspace/working/sd-scripts/requirements.txt") as f:
    pkgs = [l.strip() for l in f
            if l.strip()
            and not l.startswith("#")
            and not l.startswith("-e")
            and l.strip() != "."
            and not any(x in l.lower() for x in
                        ["torch", "torchvision", "torchaudio", "transformers"])]

subprocess.run(["pip", "install", "-q"] + pkgs, check=False)

# ── 4. 最后装 transformers（放最后，防止被其他包依赖升级覆盖）─────────
subprocess.run(["pip", "install", "-q", "transformers==4.44.2"], check=True)
subprocess.run(["pip", "install", "-q", "xformers"], check=True)


print("✅ 全部依赖安装完成")

✅ 跳过重装 torchvision，使用 CloudGPU 默认环境


Cloning into '/workspace/working/sd-scripts'...


✅ sd-scripts 就绪
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
bigframes 2.35.0 requires rich<14,>=12.4.4, but you have rich 14.1.0 which is incompatible.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.34.3 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.6 MB/s eta 0:00:00
✅ 全部依赖安装完成


In [2]:
import os

dst = "/workspace/working/train_data/10_casketch"
os.makedirs(dst, exist_ok=True)
src = "/workspace/input/datasets/yang084/comfyui-train-data/style_concept_art"
for f in os.listdir(src):
    d = os.path.join(dst, f)
    if not os.path.exists(d):
        os.symlink(os.path.join(src, f), d)
imgs = [f for f in os.listdir(dst) if f.endswith(".png")]
txts  = [f for f in os.listdir(dst) if f.endswith(".txt")]
print(f"✅ 图片：{len(imgs)} 张，caption：{len(txts)} 个")

toml_content = """
[model_arguments]
pretrained_model_name_or_path = "/workspace/input/datasets/yang084/comfyui-models/comfyui-models/checkpoints/juggernautXL_v9.safetensors"

[dataset_arguments]
train_data_dir = "/workspace/working/train_data"
resolution = "1024,1024"
enable_bucket = true
min_bucket_reso = 512
max_bucket_reso = 1024
bucket_reso_steps = 64
caption_extension = ".txt"

[training_arguments]
output_dir = "/workspace/working/lora_output"
output_name = "death_stranding_style"
save_every_n_epochs = 2
max_train_epochs = 10
learning_rate = 1e-4
network_module = "networks.lora"
network_dim = 16
network_alpha = 8
mixed_precision = "fp16"
gradient_checkpointing = true
cache_latents = true
xformers = true
"""
os.makedirs("/workspace/working/lora_output", exist_ok=True)
with open("/workspace/working/train_config.toml", "w") as f:
    f.write(toml_content)
print("✅ train_config.toml 写入完成")

✅ 图片：57 张，caption：57 个
✅ train_config.toml 写入完成


In [3]:
import subprocess

# 不用 check=True，让错误信息完整打印出来
result = subprocess.run([
    "python3", "/workspace/working/sd-scripts/sdxl_train_network.py",
    "--config_file", "/workspace/working/train_config.toml"
])
if result.returncode != 0:
    print(f"\n❌ 训练退出码: {result.returncode}，请查看上方报错信息")
else:
    print("✅ 训练完成")

2026-04-04 12:48:49.609734: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775306930.065501      91 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775306930.173395      91 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775306931.185627      91 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775306931.185669      91 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775306931.185672      91 computation_placer.cc:177] computation placer alr

accelerator device: cuda


100%|██████████| 57/57 [00:45<00:00,  1.25it/s]
2026-04-04 12:51:17 INFO     create LoRA network. base dim (rank):   lora.py:935
                             16, alpha: 8                                       
                    INFO     neuron dropout: p=None, rank dropout:   lora.py:936
                             p=None, module dropout: p=None                     
                    INFO     create LoRA for Text Encoder 1:        lora.py:1027
2026-04-04 12:51:18 INFO     create LoRA for Text Encoder 2:        lora.py:1027
                    INFO     create LoRA for Text Encoder: 264      lora.py:1035
                             modules.                                           
                    INFO     create LoRA for U-Net: 722 modules.    lora.py:1043
                    INFO     enable LoRA for text encoder: 264      lora.py:1084
                             modules                                            
                    INFO     enable LoRA for U-Net: 722 modul

import network module: networks.lora
prepare optimizer, data loader etc.
override steps. steps for 10 epochs is / 指定エポックまでのステップ数: 5700
running training / 学習開始
  num train images * repeats / 学習画像の数×繰り返し回数: 570
  num validation images * repeats / 学習画像の数×繰り返し回数: 0
  num reg images / 正則化画像の数: 0
  num batches per epoch / 1epochのバッチ数: 570
  num epochs / epoch数: 10
  batch size per device / バッチサイズ: 1
  gradient accumulation steps / 勾配を合計するステップ数 = 1
  total optimization steps / 学習ステップ数: 5700

epoch 1/10



steps:   0%|          | 0/5700 [00:00<?, ?it/s]2026-04-04 12:51:41 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 1                         
2026-04-04 12:51:41 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 1                         
2026-04-04 12:51:41 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 1                         
2026-04-04 12:51:41 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 1                         
steps:  10%|█         | 570/5700 [21:41<3:15:11,  2.28s/it, avr_loss=0.0722]

...


saving checkpoint: /workspace/working/lora_output/death_stranding_style-000008.safetensors

epoch 9/10



2026-04-04 15:45:35 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 9                         
2026-04-04 15:45:35 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 9                         
2026-04-04 15:45:35 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 9                         
2026-04-04 15:45:35 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 9                         
steps:  90%|█████████ | 5130/5700 [3:15:29<21:43,  2.29s/it, avr_loss=0.0667]


epoch 10/10



2026-04-04 16:07:19 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 10                        
2026-04-04 16:07:19 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 10                        
2026-04-04 16:07:19 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 10                        
2026-04-04 16:07:19 INFO     epoch is incremented.             train_util.py:787
                             current_epoch: 0, epoch: 10                        
steps: 100%|██████████| 5700/5700 [3:37:17<00:00,  2.29s/it, avr_loss=0.0645]



saving checkpoint: /workspace/working/lora_output/death_stranding_style.safetensors
✅ 训练完成
